# Test Stage 11 — The parameter sequencer

The paper's precision technique is a *timed* sequence: detune the resonator so the ion is
decoupled, fire a pulse that sets a starting phase, let it evolve for an exact time, re-couple,
read the phase back. The answer is only as good as the timing.

Stage 9 measured what software on this board can manage: a **45 ms** control interval, with
jitter to match. The fabric clock is **8 ns**. Five orders of magnitude separate them, so timed
switching cannot live in Python — it lives in a small state machine instead, which rewrites the
phase shifter's two coefficients at exact times or on a trigger edge.

This stage checks that it does.

### What was added to the hardware

| | |
|---|---|
| `red_pitaya_sequencer.v` | Eight parameter sets, a 32-bit down-counter, a trigger input. Slot 7, base `0x40700000` |
| `red_pitaya_phaseshift.v` | Three new inputs and a **registered** two-input mux on the coefficient pair |

The mux is registered on purpose. Putting a select directly in front of a multiplier adds logic
to a path that already reaches a hardware multiplier, and this file has lost time twice to
exactly that. Between two flip-flops it has a whole clock to settle, and it costs one cycle of
delay on a *coefficient* change — which is a control value, not signal.

**Idle at power-on.** The source bit defaults to 0, so the phase shifter uses its own registers
and the board behaves exactly as before. Step 3 checks that claim rather than asserting it.

---

### Simulation or the real trap?

**Unchanged.** The sequencer counts clocks; it neither knows nor cares what the plant is. This
stage runs against the emulated coil because that gives a known response to switch *between* —
on a real coil, use two settings from the Stage 10 atlas instead.

One caveat that outlives the emulator: **every duration here is counted in units of the 125 MHz
clock.** A clock that is wrong produces durations wrong by the same fraction, silently, and no
measurement in this notebook can detect that. See Test Stage 14a.

## What you need

| | |
|---|---|
| Stages 1–10 | Passing |
| Bitstream | Rebuilt with the sequencer — Step 0 checks and refuses otherwise |
| Time | About two minutes |
| Cables | **None** for Steps 0–3. Step 2's edge test needs a jumper between two expansion pins |

## Setup — drivers, capture, the sweep, and the sequencer

In [ ]:
import os, mmap, struct, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FS_CLK = 125e6
F0     = 40.0e6          # the coil being emulated: ~40 MHz
Q_SET  = 180.0           # ~180 at room temperature
SETTLE = 0.02
N_AVG  = 32

# =============================================================================
#  Environment shims. This board carries an older numpy and matplotlib than a
#  current desktop, and two conveniences used below arrived later than what is
#  installed here: nan_to_num's `nan=` keyword needs numpy >= 1.17, and
#  Axes.set_facecolor needs matplotlib >= 2.0. Provided once, rather than as
#  version tests scattered through the analysis cells.
# =============================================================================
def nan_fill(a, value):
    """np.nan_to_num(a, nan=value), for numpy versions without the keyword."""
    out = np.array(a, dtype=float, copy=True)
    out[~np.isfinite(out)] = value
    return out


def set_bg(ax, colour):
    """Axes background colour, across the matplotlib 1.x -> 2.x rename."""
    if hasattr(ax, 'set_facecolor'):
        ax.set_facecolor(colour)
    else:
        ax.set_axis_bgcolor(colour)


# =============================================================================
#  Drivers - the Stage 8 set, with the resonator gaining its two noise knobs.
#  CRITICAL RULE throughout: one 32-bit register access at a time.
# =============================================================================
class Bus(object):
    def __init__(self, base, span=0x1000):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, span, offset=base)
    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

def s32(v):
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(Bus):
    """Acquisition system as narrowband voltmeter (PEEK at DC) and as
    recorder (capture to memory - defined in the next cell)."""
    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = 10
    def start(self, dec_log2=10):
        self.dec = dec_log2
        self.w(0x00, 0)
        self.w(0x08, dec_log2)
        self.w(0x0C, 0)
        self.w(0x00, 1 | 2)
        self.w(0x00, 1)
    def tune(self, f):
        self.w(0x04, int(round(f / FS_CLK * (1 << 32))) & 0xFFFFFFFF)
    def measure(self, navg=N_AVG):
        zi = 0.0; zq = 0.0
        for _ in range(navg):
            zi += s32(self.r(0x18))
            zq += s32(self.r(0x1C))
        return complex(zi / navg, zq / navg)
    def f_out(self):
        return FS_CLK / (1 << self.dec)


class Resonator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None; self.q_q = None
    def config(self, f0, Q, delay=15, oshift=7):
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r  = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        rq        = np.hypot(ci, cs) / 131072.0
        self.f0_q = np.arctan2(cs, ci) / (2.0 * np.pi) * FS_CLK
        self.q_q  = np.pi * (self.f0_q / FS_CLK) / (1.0 - rq)
    def shift_f0(self, df, Q=None):
        """Retune the emulated coil by df hertz - the drift the servo fights.
        One coefficient rewrite; Q is preserved."""
        self.config(self.f0_q + df, Q if Q is not None else self.q_q,
                    delay=15, oshift=7)
    # --- the two noise knobs (registers added for this stage) -------------
    def noise_tank(self, shift=None):
        """Johnson noise INTO the loop -> Lorentzian bump. None = off.
        Each shift step is 6 dB down from raw (~9460 counts rms)."""
        self.w(0x20, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def noise_out(self, shift=None):
        """Amplifier noise AFTER the loop -> flat floor. None = off."""
        self.w(0x24, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def enable(self):  self.w(0x00, 1)
    def disable(self): self.w(0x00, 0)
    def clear(self):   self.w(0x00, 3)
    def clipped(self): return bool(self.r(0x14) & 1)
    def peek(self):    return s32(self.r(0x18)), s32(self.r(0x1C))


class PhaseShift(Bus):
    LATENCY = 2
    COEF_FS = 16384.0                 # coefficient full scale -> gain LSB
    def __init__(self):
        Bus.__init__(self, 0x40300000)
    def config_delay(self, f):
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000:
                continue
            frac  = D - np.floor(D)
            score = min(frac, 1.0 - frac)
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        self.w(0x08, best[1])
        self.w(0x0C, int(round(best[2] * 16384)))
        self.w(0x10, 0)
        return best[1], best[2]
    def set(self, A, phi_deg, f=F0):
        phi = phi_deg - 360.0 * self.LATENCY * f / FS_CLK
        self.w(0x00, int(round(A * np.cos(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
        self.w(0x04, int(round(A * np.sin(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
    def gain_lsb(self, phi_deg, f=F0):
        """Smallest change in A this actuator can express at this phase. The
        two coefficients quantise independently, so the step in |A| is set by
        the LARGER of the two projections. Step 5c compares this against the
        width of the band a software loop would need to steer within."""
        phi = np.radians(phi_deg - 360.0 * self.LATENCY * f / FS_CLK)
        return 1.0 / (self.COEF_FS * max(abs(np.cos(phi)), abs(np.sin(phi))))
    def off(self):
        self.set(0.0, 0.0)


class Generator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
    def load_sine(self):
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)
        self.w(0x0C, 0)
    def amp(self, frac):
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)
    def freq(self, f):
        self.w(0x10, int(round(f / FS_CLK * (1 << 30))))
    def run(self):
        self.w(0x00, 0x40)
        self.w(0x00, 0x11)
    def stop(self):
        self.amp(0.0)
        self.w(0x00, 0xC0)


acq, reso, ps, gen = Acq(), Resonator(), PhaseShift(), Generator()
print('drivers ready  (numpy %s, matplotlib %s)'
      % (np.__version__, matplotlib.__version__))
# =============================================================================
#  Sequencer driver  -  register block at 0x40700000 (sys[7])
#
#  Eight parameter sets, each holding a coefficient pair, a duration in clocks,
#  and a few flags. The state machine walks them at exact times. See
#  red_pitaya_sequencer.v for the register map.
# =============================================================================
class Sequencer(Bus):
    NSET = 8
    F_BURST, F_ARM, F_LAST, F_LOOP = 1, 2, 4, 8

    def __init__(self):
        Bus.__init__(self, 0x40700000)

    def version(self):
        return self.r(0x0C)

    def program(self, n, A, phi_deg, clocks, flags=0, f=F0):
        """Write one set. A and phi are converted the same way ps.set does, so
        a set means exactly what the same numbers mean through the phase
        shifter - including the latency correction."""
        phi = phi_deg - 360.0 * PhaseShift.LATENCY * f / FS_CLK
        ci = int(round(A * np.cos(np.radians(phi)) * PhaseShift.COEF_FS)) & 0xFFFF
        cq = int(round(A * np.sin(np.radians(phi)) * PhaseShift.COEF_FS)) & 0xFFFF
        base = 0x10 + 16 * n
        self.w(base + 0, ci)
        self.w(base + 4, cq)
        self.w(base + 8, int(clocks) & 0xFFFFFFFF)
        self.w(base + 12, int(flags) & 0xF)

    def read_set(self, n):
        base = 0x10 + 16 * n
        return (s32(self.r(base + 0)), s32(self.r(base + 4)),
                self.r(base + 8), self.r(base + 12))

    def trigger_cfg(self, enable=False, falling=False):
        self.w(0x08, (1 if falling else 0) | (2 if enable else 0))

    def source(self, on):
        """True: the sequencer drives the phase shifter. False: software does."""
        self.w(0x00, 4 if on else 0)

    def arm(self):
        self.w(0x00, 1 | (self.r(0x00) & 4))

    def abort(self):
        self.w(0x00, 2 | (self.r(0x00) & 4))

    def status(self):
        v = self.r(0x04)
        return {'set': v & 7, 'running': bool(v & 8),
                'done': bool(v & 16), 'trig_seen': bool(v & 32)}


seq = Sequencer()
print('sequencer driver ready')

In [ ]:
# =============================================================================
#  Memory capture - the Stage 6/7 system, condensed. Noise SPECTRA need time
#  records, and time records need memory; the PEEK voltmeter cannot see a
#  spectrum. Same safeguards as Stage 7: region from the device tree, stop
#  address inclusive, poison pre-fill, write-error and poison checks that
#  RAISE rather than return plausible garbage. (Bitstreams from the Stage 8
#  rebuild onward carry the trusted badge, so TrustZone no longer matters -
#  but the checks stay, because they cost nothing and catch everything.)
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]

def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None

blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert blob and len(blob) >= 8, 'cannot read the device tree memory node'
declared_end = _be32(blob, 0) + _be32(blob, 1)
assert declared_end < 512 * 1024**2, 'no memory above the declared end - see Stage 6 Step 1'
MEM_START = (declared_end + 0xFFF) & ~0xFFF
MEM_STOP  = MEM_START + (0x00800000 & ~7) - 8       # LAST address, inclusive
POISON    = 0xDEADBEEF
print('capture region 0x%08X .. 0x%08X inclusive' % (MEM_START, MEM_STOP + 7))


CAPTURE_TRACE = None      # set to a path (e.g. '/tmp/capture.log') to log every
                          # capture to disk before it runs


def _trace(msg):
    """Append to the trace file and force it to disk.

    A dead kernel takes buffered stdout with it, which is why a crash inside
    capture() leaves the notebook showing nothing at all. This survives,
    so the last line in the file names the call that killed it."""
    if not CAPTURE_TRACE:
        return
    with open(CAPTURE_TRACE, 'a') as fh:
        fh.write('%.3f %s\n' % (time.time(), msg))
        fh.flush()
        os.fsync(fh.fileno())


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close(); os.close(fd)


def capture(f_lo, dec_log2, n_samples, settle=SETTLE):
    """Record n_samples of the complex stream; raises on any transport fault."""
    _trace('capture f=%.1f dec=%d n=%d' % (f_lo, dec_log2, n_samples))
    _poison(n_samples * 8)
    acq.w(0x00, 0)
    acq.w(0x08, dec_log2); acq.dec = dec_log2
    acq.w(0x0C, 0)
    acq.tune(f_lo)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)
    # Read the bounds back BEFORE arming. The writer is a DMA master: if the
    # stop bound did not take - wrong bitstream, unimplemented register, a
    # write that silently went nowhere - it runs from MEM_START upward through
    # whatever follows, and what follows is the memory Linux is using. That is
    # not an exception with a traceback to read; it is a dead kernel. Two
    # register reads cost microseconds and turn the worst failure mode in this
    # notebook into a sentence.
    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)                              # ensure nothing is armed
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded DMA writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples (%d bytes) exceeds the %d-byte '
                         'capture region'
                         % (n_samples, n_samples * 8, MEM_STOP + 8 - MEM_START))
    # SETTLE FIRST, RECORD SECOND. Arming immediately and sleeping afterwards
    # would let the record's opening milliseconds hold the filter pipeline still
    # flushing the PREVIOUS tuning - which in a sweep poisons each point with its
    # predecessor's power, most visibly the first point, and can drag a fit wide
    # by tens of percent from the far end of the grid. Instead the chain
    # free-runs at the new tuning while the pipeline flushes, and only then does
    # the writer start: sample 0 of every record is already settled.
    acq.w(0x00, 1 | 2)                              # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                              # ...flush the pipeline...
    acq.w(0x00, 1 | 16 | 128)                       # ...then arm, fresh writer
    acq.w(0x00, 1 | 16)
    # WAIT FOR THE WRITER; DO NOT GUESS AT IT.
    #
    # This used to be one sleep of "as long as the record should take, plus
    # 5 ms", after which the writer was disarmed and memory read. Two things
    # were wrong with that, and both showed up on 2026-08-04 as an intermittent
    # 'unwritten words in the record' on captures that were perfectly healthy:
    #
    #  1. DMA_COUNT (0x34) was not consulted until AFTER the decision to stop
    #     had already been made. The sleep was a guess with no verification.
    #  2. DMA_COUNT reports samples HANDED TO the memory writer, not samples
    #     that have landed in DDR. The writer bursts, so the last burst can
    #     still be in flight when the count already reads complete.
    #
    # A fixed 5 ms of margin is also the wrong shape: it is 30% of a short
    # record and 0.1% of a long one, so the failure is rare, load-dependent and
    # looks like a hardware fault. It is not - the record was read a few
    # hundred microseconds early.
    #
    # Poll until the record is complete, then let the last burst land.
    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5         # generous, not 5 ms
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                                  # disarm
    time.sleep(0.002)                               # last burst reaches DDR

    st  = acq.r(0x38)
    if st & 8:
        raise RuntimeError('memory refused writes - see Stage 6; run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    # try/finally, because everything between the open and the close can raise:
    # mmap itself, the slice (SIGBUS on a bad physical address), an allocation
    # failure on a tight board. Without it each failure leaks a file descriptor
    # AND a mapping of /dev/mem - and Stage 10's scan loop swallows per-point
    # exceptions by design, so the leak would repeat silently across hundreds
    # of grid points until the process runs out of descriptors or address
    # space. That surfaces later as an unrelated-looking out-of-memory error.
    def _read_region(nbytes):
        fd = mm = None
        try:
            fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
            mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
            return mm[0:nbytes]
        finally:
            if mm is not None:
                mm.close()
            if fd is not None:
                os.close(fd)

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    _limit = max(8, 0.001 * u32.size)
    if got and int(np.count_nonzero(u32 == POISON)) > _limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms; if it was
        # merely late it will not. Retrying once separates the two, which the
        # single-shot check could not: it reported a transport fault for both.
        time.sleep(0.02)
        buf = _read_region(got * 8)
        u32 = np.frombuffer(buf, dtype='<u4')
        n_bad = int(np.count_nonzero(u32 == POISON))
        if n_bad > _limit:
            raise RuntimeError(
                'unwritten words in the record - transport fault, see Stage 6 '
                '(%d of %d words still unwritten 20 ms after the writer '
                'reported the record complete)' % (n_bad, u32.size))
    raw = np.frombuffer(buf, dtype='<i4').astype(np.float64)
    acq.w(0x00, 1)                                  # back to free-running voltmeter
    _trace('   ok got=%d' % got)
    return raw[0::2] + 1j * raw[1::2]


def welch_psd(z, seg=2048):
    """Averaged power spectrum of a complex record - the standard way to see
    noise: many short transforms averaged beat one long one, because noise in
    a single transform is 100% uncertain no matter how long it is."""
    k = len(z) // seg
    w = np.hanning(seg)
    p = np.zeros(seg)
    for i in range(k):
        p += np.abs(np.fft.fft(z[i * seg:(i + 1) * seg] * w)) ** 2
    f = np.fft.fftfreq(seg, 1.0 / acq.f_out())
    order = np.argsort(f)
    return f[order], p[order] / max(k, 1)


def _lorentz_core(f, p, guard=0.05, minpts=3):
    """Centre, width, peak and BASELINE of a Lorentzian power curve.

    1/p is a parabola in f only for a pure Lorentzian. Any additive floor -
    detector noise, an amplifier pedestal - is absorbed into that parabola and
    inflates the fitted width. That is negligible where the peak towers over
    the floor and severe where it does not, because peak power falls as the
    square of the Q ratio: a floor worth 2% of the open-loop peak is worth 32%
    of a 4x-broadened one. Uncorrected bias, measured: +18% in linewidth at
    Q_eff/Q0 = 1, +26% at 0.5, +153% at 0.25.

    The floor is found by search, not from the wings. For each trial floor:
    subtract it, fit, rebuild the full Lorentzian-plus-floor, and keep the
    trial that reproduces the measured points best. A wing estimate needs
    points beyond ~1.5 linewidths, which a broad line does not have on this
    grid - exactly the case that needs correcting most. The search uses every
    point, so it works at any width. Two passes: coarse over [0, min(p)], then
    fine around the winner.

    Returns (f0, linewidth, peak_above_baseline, baseline).
    """
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                    # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        mod = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        return float(np.sqrt(np.mean(((p - mod) / pmax) ** 2))), (f0, lw, p0, b)

    hi = 0.999 * float(np.min(p))
    best, best_b = _try(0.0), 0.0           # b = 0 is the uncorrected fit
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        raise ValueError('fit window too narrow: %d points'
                         % int(np.count_nonzero(p > guard * p.max())))
    f0, lw, p0, base = best[1]
    return f0, lw, p0, base


def fit_lorentzian_psd(f, p, guard=0.05):
    """Centre, width from a power spectrum: 1/p is a parabola (Stage 8), after
    the noise floor has been taken out of p."""
    f0, lw, _, _ = _lorentz_core(f, p, guard=guard, minpts=3)
    return f0, lw

In [ ]:
# ============================ CONFIGURATION ==================================
EMULATION  = True    # True: digital resonator, known answers, PASS gates.
                     # False: the real coil - same scan, gates become measurements

TANK_SH    = 3       # tank (Johnson) noise level, 6 dB per step
OUT_SH     = 4       # amplifier-floor level (off during sweeps automatically)

# The gain axis must BRACKET the self-oscillation boundary, or there is no
# boundary in the atlas and every stability margin the designer prints is
# meaningless. K rises ~0.73 per unit A, so onset is near A = 1.4 (+3 dB).
GAINS_DB   = np.arange(-26.0, 6.01, 2.0)     # feedback gain axis (dB of A)
PHIS       = np.arange(0.0, 360.0, 15.0)     # feedback phase axis: 24 points
SWEEP_PTS  = 15      # frequencies per mini-sweep at each grid point
SWEEP_SPAN = 2.5     # mini-sweep half-width in open-loop linewidths
SWEEP_N    = 2048    # samples per frequency point (halved at deep decimations)
SETTLE     = 0.02

# A point is OSCILLATING if, with the noise drive removed, it still holds this
# fraction of its driven power. A ratio, so it needs no absolute floor - on a
# digital plant the undriven floor is exactly zero.
SUS_OSC    = 0.15
DRIVE_STEP = 2       # noise-shift steps (12 dB) used to test loop linearity
# Grid cost: 17 x 24 points x ~1 s. About 7 minutes.
# =============================================================================

def coherence(z):
    """1 for pure noise, 0 for a pure tone; equals 1 - f^2 where f is the
    power fraction carried by a coherent tone. Reported as a diagnostic.
    Not used to classify points: for narrowband noise the effective sample
    count is small and this scatters ~30%, far too coarse to detect anything
    subtle."""
    p = np.abs(z) ** 2
    m = float(np.mean(p))
    return float(np.var(p)) / max(m * m, 1e-300)


def tone_fraction(R):
    return float(np.sqrt(max(0.0, 1.0 - R)))


def fit_lorentz3(f, p, guard=0.10):
    """Centre, width, peak and baseline of a Lorentzian power curve (the
    Stage 9 method, with the floor removed - see _lorentz_core in the cell
    above). Raises if the guard leaves too few points to determine a parabola;
    silence there is how a thin fit window becomes a huge Q.

    The peak is now the height ABOVE the baseline, so (pi/2)*p0*lw is the area
    of the bump alone - which is what T_eff wants. The baseline is returned so
    the caller can rebuild the full curve when testing the fit."""
    return _lorentz_core(f, p, guard=guard, minpts=5)


def _win_half(dec):
    """The receiver's listening half-width at decimation 2^dec."""
    return 0.27 * FS_CLK / (1 << dec)


def _dec_for(spacing):
    """Match the receiver to the grid: window ~ half the point spacing,
    rounded WIDER. A too-wide window smears the width, which the
    deconvolution below undoes exactly; a too-narrow one leaves gaps a narrow
    line can hide in, which no amount of fitting can undo."""
    d = int(np.floor(np.log2(0.27 * FS_CLK / (0.5 * spacing))))
    return int(np.clip(d, 8, 14))


def _model(fs, dec, f0, lw):
    """A width-lw Lorentzian at f0 seen through this receiver on this grid:
    the window-averaged (smeared) curve, in closed form."""
    W = _win_half(dec)
    return ((lw / 2.0) * (np.arctan(2.0 * (fs + W - f0) / lw)
                        - np.arctan(2.0 * (fs - W - f0) / lw)) / (2.0 * W))


def _deconv_lw(lw_m, f0, fs, dec):
    """Undo the window smear with no calibration constants: bisect the true
    width whose model, pushed through the SAME window, grid and fit,
    reproduces the measured width.

    Returns (lw_true, bracketed). `bracketed` means the search hit its lower
    bound - the correction wanted to be more than 4x and was truncated, which
    would inflate Q with no other symptom. The caller discards those."""
    def fwd(lw_t):
        # The guard in fit_lorentz3 protects fits to real DATA. Tripping it on
        # our own noise-free model just means the trial width is too narrow for
        # this grid to express, which is the same as "narrower than measured".
        try:
            return fit_lorentz3(fs, _model(fs, dec, f0, lw_t) + 1e-12)[1]
        except ValueError:
            return 0.0
    lo, hi = lw_m / 4.0, lw_m * 1.05
    if fwd(hi) < lw_m:
        return lw_m, False               # smear negligible at this geometry
    for _ in range(30):
        mid = 0.5 * (lo + hi)
        if fwd(mid) < lw_m:
            lo = mid
        else:
            hi = mid
    out = 0.5 * (lo + hi)
    return out, bool(out < 1.02 * lw_m / 4.0)


def noise_sweep(centre, pts=SWEEP_PTS):
    """Adaptive power sweep. Each pass matches the receiver to the grid, then
    reads the data before trusting any fit:
      - low contrast          -> structure wider than the span: widen, retry
      - one point owns the power -> narrower than the grid: zoom on it
      - otherwise fit, accepting only if the centre is inside the span, the
        width is resolvable here, and the curve goes through the points
    An accepted width is deconvolved from the receiver window. Five passes
    maximum; anything still unresolved returns nans - reported, never stored.

    Returns (f0, lw_true, p0, area, coh). Area uses the MEASURED (smeared)
    peak and width: window averaging conserves the integral, so that pair
    gives the true area directly, while Q needs the deconvolved width."""
    span = SWEEP_SPAN * LW0
    for _ in range(5):
        fs = centre + np.linspace(-span, span, pts)
        spacing = fs[1] - fs[0]
        dec = _dec_for(spacing)
        nsamp = SWEEP_N if dec <= 12 else SWEEP_N // 2
        p  = np.empty(pts); ch = np.empty(pts)
        for k, fq in enumerate(fs):
            z = capture(fq, dec, nsamp)
            p[k]  = float(np.var(z))
            ch[k] = coherence(z)
        coh   = float(ch[int(np.argmax(p))])
        base  = float(np.min(p))
        share = (p.max() - base) / max(float(np.sum(p - base)), 1e-300)
        contrast = p.max() / max(float(np.median(p)), 1e-300)
        if contrast < 2.0:
            span = min(span * 3.0, 8.0 * LW0)
            continue
        if share > 0.6:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
            continue
        try:
            f0, lw, p0, pbase = fit_lorentz3(fs, p)
        except ValueError:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
            continue
        inside = bool(fs[0] < f0 < fs[-1])
        # The baseline must be added back before judging the fit: comparing
        # data-with-floor against a floor-free Lorentzian rejects a CORRECT fit
        # whenever the floor matters, which is exactly the broad-line case the
        # baseline removal was added for.
        fitc   = p0 / (1.0 + (2.0 * (fs - f0) / lw) ** 2) + pbase
        resid  = float(np.sqrt(np.mean(((p - fitc) / p.max()) ** 2)))
        if inside and 1.5 * spacing <= lw <= (fs[-1] - fs[0]) and resid < 0.15:
            area = (np.pi / 2.0) * p0 * lw
            lw_t, bracketed = _deconv_lw(lw, f0, fs, dec)
            if bracketed:
                return np.nan, np.nan, np.nan, np.nan, coh
            return f0, lw_t, p0, area, coh
        if inside and spacing / 5.0 < lw < 1.5 * spacing:
            centre, span = f0, max(2.0 * lw, 5e2)
        else:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
    return np.nan, np.nan, np.nan, np.nan, np.nan        # honest failure

---
## Step 0 — Is the sequencer actually in the chip?

An unused bus slot answers every read with zero and looks politely dead, so a bitstream without
the sequencer would let this whole notebook run and produce plausible nonsense. The module
carries a fixed version number for exactly this reason: read it first, and refuse to continue
if it does not come back.

In [ ]:
EXPECT_VERSION = 0x5E900001

v = seq.version()
print('sequencer version register: 0x%08X' % v)
if v != EXPECT_VERSION:
    print()
    print('*** THIS BITSTREAM HAS NO SEQUENCER ***')
    print('    Slot 7 answered 0x%08X, not 0x%08X. Either this is an older' % (v, EXPECT_VERSION))
    print('    build, or the bitstream did not load. Rebuild and reload:')
    print('      PC, Vivado Tcl console:  source C:/rp/prj/v0.94/Test_Project/build_project.tcl')
    print('    then check the build fingerprint with check_system.py.')
    raise SystemExit('stopping: sequencer absent')

# Registers must hold what is written to them - a module that answers reads
# from thin air would pass the version check and fail everything after it.
seq.program(3, 0.25, 90.0, 1234, flags=Sequencer.F_BURST)
ci, cq, dur, fl = seq.read_set(3)
ok_rw = (dur == 1234 and fl == Sequencer.F_BURST and ci != 0 and cq != 0)
print('set 3 read back: ci %+6d  cq %+6d  duration %d  flags 0x%X' % (ci, cq, dur, fl))

# Does a programmed set actually BECOME the set the machine runs?
#
# The read-back above writes set 3 and reads set 3, and it passed on 2026-08-03
# while the hardware was decoding every set one slot high and wrapping set 7 onto
# set 0. It would pass, and so would any read-back test, however thorough: reads
# used the same faulty decode as writes, so both went to the wrong slot and
# agreed with each other. The bus was internally consistent - it had simply
# relabelled every set.
#
# What broke the symmetry was the state machine, which reads mem[cur] by index
# and never goes through that decode. So the only test that can see this is a
# BEHAVIOURAL one: program a short two-set sequence, run it, and check the
# machine ends where it was told to end.
#
# Left under software control (source False), so this disturbs nothing.
seq.abort()
seq.program(0, 0.0, 0.0, 1250, flags=0)                 # 10 us
seq.program(1, 0.0, 0.0, 1250, flags=Sequencer.F_LAST)  # 10 us, then stop
seq.trigger_cfg(enable=False)
seq.source(False)
seq.arm()
time.sleep(0.01)                                        # 1000x the 20 us it needs
stq = seq.status()
ok_run = (stq['done'] and not stq['running'] and stq['set'] == 1)
print('a two-set sequence runs and stops at set 1 : %s  (set %d, running %s, done %s)'
      % ('yes' if ok_run else 'NO', stq['set'], stq['running'], stq['done']))
if not ok_run:
    print('   Stuck at set 0 and still running means mem_dur[0] was never')
    print('   written - the set index is decoded wrongly in the RTL.')
seq.abort()

seq.source(False)                 # leave the phase shifter under software control
seq.abort()
st = seq.status()
print('status: set %d, running %s, done %s' % (st['set'], st['running'], st['done']))

# The plant, so there is something to watch the sequencer switch between.
reso.config(F0, Q_SET)
ps.config_delay(F0); ps.off()
gen.load_sine(); gen.amp(0.0); gen.run()
reso.enable(); reso.clear()
reso.noise_tank(TANK_SH); reso.noise_out(None)
acq.start(dec_log2=10); acq.tune(reso.f0_q)
time.sleep(0.2)
LW0 = reso.f0_q / reso.q_q
print()
print('STEP 0:', 'PASS' if (ok_rw and ok_run) else 'FAIL - registers do not hold what is written')

---
## Step 1 — Does a duration mean what it says?

The obvious test — arm the sequencer, start a capture, see where the switch lands — has a
problem: arming and capturing are two separate software writes, milliseconds apart and jittery,
so the *absolute* position of the edge says more about Python than about the counter.

So measure a **period** instead. Two sets that loop forever produce a square wave in the
captured amplitude, and its period is `(D0 + D1)` clocks no matter when the capture started.
That is immune to the software skew entirely, and it tests the counter, the set advance, and
the loop flag in one measurement.

Set 0 leaves the feedback off; set 1 applies a strong narrowing gain, which lifts the
resonator's amplitude. The switch between them is what shows up in the record.

In [ ]:
PHI_N   = 210.0          # a narrowing phase; any strong setting will do
A_ON    = 0.6
DEC_SEQ = 10                                   # capture decimation
F_OUT   = FS_CLK / (1 << DEC_SEQ)

def run_square(d0_clocks, d1_clocks, nsamp=8192):
    """Loop two sets forever and capture. Returns the record."""
    seq.abort()
    seq.program(0, 0.0,  0.0,   d0_clocks, flags=0)
    seq.program(1, A_ON, PHI_N, d1_clocks, flags=Sequencer.F_LAST | Sequencer.F_LOOP)
    seq.trigger_cfg(enable=False)
    seq.source(True)                           # sequencer now drives the coefficients
    seq.arm()
    time.sleep(0.05)                           # let it settle into the loop
    z = capture(reso.f0_q, DEC_SEQ, nsamp)
    seq.abort(); seq.source(False); ps.off()
    return z


def measure_period(z, f_out):
    """Period of the amplitude square wave, from its strongest spectral line.

    The envelope is |z|, whose mean is removed so the line being found is the
    switching, not the carrier."""
    env = np.abs(z)
    env = env - env.mean()
    w = np.hanning(len(env))
    P = np.abs(np.fft.rfft(env * w)) ** 2
    P[0] = 0.0
    k = int(np.argmax(P))
    if k == 0:
        return np.nan
    # Parabolic interpolation on the peak, so the answer is not quantised to a
    # whole bin. At a maximum the curvature (a - 2b + c) is NEGATIVE - clamping
    # it to a small positive number sends the correction to infinity and the
    # period to zero, which is exactly what it did before this was fixed. Guard
    # against a flat peak instead, and keep the shift inside the bin it belongs
    # to.
    kf = float(k)
    if 0 < k < len(P) - 1:
        a, b, c = P[k-1], P[k], P[k+1]
        den = a - 2.0 * b + c
        if den < 0.0:
            kf = k + float(np.clip(0.5 * (a - c) / den, -0.5, 0.5))
    f_sw = kf * f_out / len(env)
    return 1.0 / f_sw if f_sw > 0 else np.nan


print('%14s %16s %16s %10s' % ('programmed', 'expected [us]', 'measured [us]', 'error'))
rows, worst = [], 0.0
for total_us in (200.0, 500.0, 1000.0, 2000.0):
    total = int(total_us * 1e-6 * FS_CLK)
    d0 = total // 2
    d1 = total - d0
    z = run_square(d0, d1)
    per = measure_period(z, F_OUT)
    if not np.isfinite(per):
        print('%14.0f %16.1f %16s %10s' % (total, total_us, 'no line', 'FAIL'))
        rows.append(False); continue
    err = 100.0 * (per * 1e6 / total_us - 1.0)
    worst = max(worst, abs(err))
    rows.append(abs(err) < 2.0)
    print('%14d %16.1f %16.1f %9.2f%%' % (total, total_us, per * 1e6, err))

print()
print('worst period error: %.2f%%' % worst)
print('STEP 1:', 'PASS' if all(rows) and worst < 2.0 else
      'FAIL - a programmed duration is not the duration produced')

---
## Step 2 — Waiting, and the trigger

A set with a duration of zero does not run for zero time — it **waits for a trigger**. That is
how the sequence is made to start on an external event rather than on a software write.

Two things can be checked with no cable at all: a sequence that reaches a waiting set must stop
there and stay stopped, and it must not claim to have seen a trigger that never arrived. The
edge itself needs a jumper between two pins on the expansion header, so that part is optional
and clearly marked.

In [ ]:
# --- 2a. a waiting set must actually wait ----------------------------------
seq.abort()
seq.program(0, 0.0,  0.0,   int(0.001 * FS_CLK), flags=0)          # 1 ms
seq.program(1, A_ON, PHI_N, 0,                   flags=0)          # wait for trigger
seq.program(2, 0.0,  0.0,   int(0.001 * FS_CLK), flags=Sequencer.F_LAST)
seq.trigger_cfg(enable=False)
seq.source(True)
seq.arm()

time.sleep(0.2)                        # far longer than set 0's 1 ms
st1 = seq.status()
time.sleep(0.2)
st2 = seq.status()

print('after 200 ms : set %d, running %s, done %s, trigger seen %s'
      % (st1['set'], st1['running'], st1['done'], st1['trig_seen']))
print('after 400 ms : set %d, running %s, done %s, trigger seen %s'
      % (st2['set'], st2['running'], st2['done'], st2['trig_seen']))

ok_wait = (st1['set'] == 1 and st2['set'] == 1
           and st1['running'] and st2['running']
           and not st1['done'] and not st2['done']
           and not st2['trig_seen'])
print('waits at the trigger set, indefinitely, and reports no trigger:',
      'yes' if ok_wait else 'NO')

# --- 2b. abort must release it ---------------------------------------------
seq.abort()
time.sleep(0.02)
st3 = seq.status()
ok_abort = st3['done'] and not st3['running']
print('abort stops it                                          :',
      'yes' if ok_abort else 'NO')

# --- 2c. the edge itself (needs a jumper) ----------------------------------
#  DIO0_P is the trigger input. To test the edge, jumper it to a pin you can
#  drive and toggle that pin. Without the jumper this reports SKIPPED rather
#  than passing something it never measured.
TRIGGER_JUMPER_FITTED = False

if TRIGGER_JUMPER_FITTED:
    seq.abort()
    seq.program(0, 0.0,  0.0,   0, flags=0)                        # wait immediately
    seq.program(1, A_ON, PHI_N, int(0.001 * FS_CLK), flags=Sequencer.F_LAST)
    seq.trigger_cfg(enable=True, falling=False)
    seq.source(True); seq.arm()
    time.sleep(0.05)
    before = seq.status()
    # drive the jumpered output pin here, then:
    time.sleep(0.05)
    after = seq.status()
    ok_edge = (not before['trig_seen']) and after['trig_seen']
    print('trigger edge advances the sequence                      :',
          'yes' if ok_edge else 'NO')
else:
    ok_edge = None
    print('trigger edge                                            : SKIPPED')
    print('   Fit a jumper from DIO0_P to a drivable pin, set')
    print('   TRIGGER_JUMPER_FITTED = True, and add the pin toggle above.')

seq.abort(); seq.source(False); ps.off()
print()
print('STEP 2:', 'PASS' if (ok_wait and ok_abort) else
      'FAIL - the waiting or abort behaviour is wrong')

---
## Step 3 — With the sequencer idle, has anything moved?

This is the one that matters most. The sequencer added a mux to the coefficient path of the
feedback loop — the one path in this design that must never be disturbed. The claim is that with
the source bit at 0 the phase shifter behaves exactly as it did before.

That claim is testable: set the sequencer's registers to something conspicuous, leave the source
bit at 0, and measure the resonator's linewidth at a known feedback setting. If the mux is
wired correctly the sequencer's contents are invisible.

In [ ]:
def measure_lw(A, phi):
    ps.set(A, phi); reso.clear(); time.sleep(0.05)
    out = noise_sweep(reso.f0_q)
    ps.off(); reso.clear()
    return out[1]

seq.abort(); seq.source(False)

# a deliberately conspicuous set, which must have no effect while source = 0
for n in range(Sequencer.NSET):
    seq.program(n, 1.5, 30.0 * n, 1000, flags=0)

A_T, PHI_T = 0.3, PHI_N
lw_quiet = measure_lw(A_T, PHI_T)
seq.arm()                                  # running, but not driving anything
time.sleep(0.05)
lw_running = measure_lw(A_T, PHI_T)
seq.abort()

print('linewidth with the sequencer idle    : %.0f Hz' % lw_quiet)
print('linewidth with the sequencer running : %.0f Hz' % lw_running)
if np.isfinite(lw_quiet) and np.isfinite(lw_running):
    diff = abs(lw_running / lw_quiet - 1.0)
    print('difference                           : %.2f%%' % (100 * diff))
    ok_reg = diff < 0.05
else:
    ok_reg = False
    print('difference                           : one sweep did not resolve')

print()
print('STEP 3:', 'PASS' if ok_reg else
      'FAIL - the sequencer is affecting the loop while it should be invisible')
print()
print('After any rebuild, also check the timing report and re-measure the')
print('feedback loop delay: it must still be 119.87 ns.')

---
## Stage 11 summary

| Step | What it proved |
|---|---|
| 0 | The sequencer is in the chip and its registers hold what is written |
| 1 | A programmed duration produces that duration, measured as a period so software timing cannot contaminate it |
| 2 | A zero duration waits for a trigger and waits indefinitely; abort releases it |
| 3 | With the source bit at 0 the feedback loop is unaffected — the mux is invisible, as claimed |

### What is not yet wired

`burst_o` and `acq_arm_o` leave the sequencer but connect to nothing. Wiring them needs new
ports on the signal generator and the acquisition system, which is a separate change. Until
then, the excitation pulse and the coherent capture start still come from software, with the
45 ms uncertainty that implies — which is fine for setting things up and not fine for the
measurement the sequencer exists to enable.

### The limit that does not go away

Everything here is counted in 125 MHz clocks. The trigger is synchronised through two
flip-flops, so expect ±8 ns of jitter on an external edge. And a clock that is *wrong* makes
every duration wrong by the same fraction with nothing to show for it — which is Stage 14a's
job, not this one's.

In [ ]:
# Board back to power-on behaviour.
seq.abort(); seq.source(False)
seq.trigger_cfg(enable=False)
reso.noise_tank(None); reso.noise_out(None)
ps.off()
gen.stop()
reso.disable()
acq.w(0x00, 0)
print('all quiet: sequencer idle, feedback open, resonator disabled')